# 01 — Data Collection
Load cached prices, compute log-returns, inspect data quality.

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

# src imports — use actual function signatures from data.py
from data import load_prices, log_returns, save_returns, load_returns
from utils import DATA_PROC, DATA_RAW, TABLES_DIR, PLOTS_DIR, set_theme, save_table
from constants import MARKET_COL, OIL_COL, FX_COL, STOCK_UNIVERSE

print('Columns expected in returns:', MARKET_COL, OIL_COL, FX_COL)
print('Stock universe:', STOCK_UNIVERSE)


Columns expected in returns: NIFTY BRENT USDINR
Stock universe: ['RELIANCE', 'ONGC', 'IOC', 'BPCL', 'INDIGO', 'HPCL', 'ADANIPORTS', 'TATAMOTORS', 'MARUTI', 'ASIANPAINT', 'HINDUNILVR', 'ITC', 'TCS', 'INFY', 'CIPLA']


## Load prices

In [2]:
# Load prices from cached parquet (no internet required).
# To refresh from yfinance, call: load_prices(force_refresh=True)
prices = load_prices(force_refresh=False)
print(f'Prices shape : {prices.shape}')
print(f'Date range   : {prices.index.min().date()} → {prices.index.max().date()}')
print(f'Columns      : {list(prices.columns)}')
prices.tail(3)


$HPCL.NS: possibly delisted; no price data found  (1d 2026-04-14 -> 2026-05-26)

1 Failed download:
['HPCL.NS']: possibly delisted; no price data found  (1d 2026-04-14 -> 2026-05-26)
$HPCL.NS: possibly delisted; no price data found  (1d 2026-04-14 -> 2026-05-26)

1 Failed download:
['HPCL.NS']: possibly delisted; no price data found  (1d 2026-04-14 -> 2026-05-26)
$HPCL.NS: possibly delisted; no price data found  (1d 2026-04-14 -> 2026-05-26)

1 Failed download:
['HPCL.NS']: possibly delisted; no price data found  (1d 2026-04-14 -> 2026-05-26)
Skipping HPCL.NS (failed after retries)
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTOR

Prices shape : (1929, 18)
Date range   : 2019-01-01 → 2026-05-25
Columns      : ['RELIANCE', 'ONGC', 'IOC', 'BPCL', 'INDIGO', 'HPCL', 'ADANIPORTS', 'TATAMOTORS', 'MARUTI', 'ASIANPAINT', 'HINDUNILVR', 'ITC', 'TCS', 'INFY', 'CIPLA', 'BRENT', 'NIFTY', 'USDINR']


,RELIANCE,ONGC,IOC,BPCL,INDIGO,HPCL,ADANIPORTS,TATAMOTORS,MARUTI,ASIANPAINT,HINDUNILVR,ITC,TCS,INFY,CIPLA,BRENT,NIFTY,USDINR
2026-05-21,1349.599976,295.850006,140.529999,296.399994,4403.000000,715.939323,1793.300049,897.490735,13010.0,2598.800049,2179.000000,308.049988,2296.067383,1181.199951,1401.900024,102.580002,23654.699219,96.528297
2026-05-22,1354.500000,290.000000,139.470001,295.600006,4438.600098,715.939323,1786.900024,897.490735,12987.0,2639.800049,2203.600098,301.700012,2286.300049,1174.500000,1399.199951,103.540001,23719.300781,96.173897
2026-05-25,1362.900024,284.850006,143.800003,306.700012,4504.899902,715.939323,1801.199951,897.490735,13123.0,2664.899902,2208.000000,303.049988,2313.399902,1173.199951,1406.400024,100.209999,23962.300781,95.315002


## Compute log returns

In [3]:
# Compute log returns  (data.py: log_returns(prices) -> pd.DataFrame)
returns = log_returns(prices)
print(f'Returns shape: {returns.shape}')
print(f'Date range   : {returns.index.min().date()} → {returns.index.max().date()}')
# Winsorize BRENT at ±25% — removes Yahoo Finance roll-date artefacts
returns['BRENT'] = returns['BRENT'].clip(-0.25, 0.25)
returns.tail(3)


Returns shape: (1928, 18)
Date range   : 2019-01-02 → 2026-05-25


,RELIANCE,ONGC,IOC,BPCL,INDIGO,HPCL,ADANIPORTS,TATAMOTORS,MARUTI,ASIANPAINT,HINDUNILVR,ITC,TCS,INFY,CIPLA,BRENT,NIFTY,USDINR
2026-05-21,-0.007456,-0.008247,0.017878,0.008981,0.031938,0.0,0.011610,0.0,0.000538,0.000077,-0.013810,0.001624,-0.000086,-0.010527,0.001713,-0.023508,-0.000182,-0.000388
2026-05-22,0.003624,-0.019972,-0.007571,-0.002703,0.008053,0.0,-0.003575,0.0,-0.001769,0.015653,0.011226,-0.020829,-0.004263,-0.005688,-0.001928,0.009315,0.002727,-0.003678
2026-05-25,0.006182,-0.017918,0.030574,0.036863,0.014827,0.0,0.007971,0.0,0.010418,0.009463,0.001995,0.004465,0.011783,-0.001108,0.005133,-0.032690,0.010193,-0.008971


## Save returns

In [4]:
# Save processed returns to data/processed/returns.parquet
save_returns(returns)
print(f'Saved returns → {DATA_PROC}/returns.parquet')

# Round-trip check
r2 = load_returns()
assert r2.shape == returns.shape, 'Shape mismatch on reload'
print('Round-trip load OK')


Saved returns → C:\Users\anves\projects\osi_clean\data\processed/returns.parquet
Round-trip load OK


## Descriptive stats

In [5]:
# Descriptive statistics (annualised)
desc = (returns.describe() * 100).round(3)
desc.loc['annualised_vol'] = (returns.std() * (252 ** 0.5) * 100).round(3)
save_table(desc.T, 'descriptive_stats')
desc.T


Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\descriptive_stats.csv


,count,mean,std,min,25%,50%,75%,max,annualised_vol
RELIANCE,192800.0,-0.125,9.283,-403.727,-0.747,0.064,0.853,5.126,147.357
ONGC,192800.0,0.030,1.598,-17.279,-0.937,0.000,1.004,7.992,25.367
IOC,192800.0,0.020,2.181,-6.575,-1.032,0.004,0.995,67.525,34.628
BPCL,192800.0,-0.018,1.712,-15.242,-1.069,-0.008,1.079,7.179,27.180
INDIGO,192800.0,0.038,2.937,-8.690,-1.295,0.000,1.307,95.962,46.619
HPCL,192800.0,0.049,1.481,-6.262,-0.717,0.000,0.898,5.554,23.513
ADANIPORTS,192800.0,0.045,2.411,-7.776,-1.112,0.000,1.103,74.070,38.268
TATAMOTORS,192800.0,0.037,1.792,-6.777,-0.919,0.000,1.048,6.836,28.444
MARUTI,192800.0,0.019,2.638,-97.873,-0.853,0.058,0.979,8.389,41.881
ASIANPAINT,192800.0,-0.009,2.986,-119.860,-0.690,0.048,0.802,4.674,47.399


## Quick visualisation

In [6]:
# Quick visualisation — sector cumulative returns
from constants import SECTOR_MAP
from event_study import get_sector_returns
sector_ret = get_sector_returns(returns)

set_theme()
fig, ax = plt.subplots(figsize=(12, 4))
for sec in ['OIL_GAS', 'AUTO', 'FMCG', 'IT', 'PHARMA']:
    ax.plot(sector_ret.index, sector_ret[sec].cumsum() * 100, label=sec, lw=1.5)
ax.set_ylabel('Cumulative log-return (%)')
ax.set_title('Sector cumulative returns — equal-weight baskets')
ax.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sector_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('NB01 complete ✓')


NB01 complete ✓
